In [4]:
library(AnnotationHub)
library(GenomicRanges)

# Question1

In [23]:
query(ah, "CpG")

AnnotationHub with 62 records
# snapshotDate(): 2026-04-23
# $dataprovider: UCSC, Haemcode
# $species: Homo sapiens, Bos taurus, Pan troglodytes, Mus musculus, Felis c...
# $rdataclass: GRanges, BigWigFile
# additional mcols(): taxonomyid, genome, description,
#   coordinate_1_based, maintainer, rdatadateadded, preparerclass, tags,
#   rdatapath, sourceurl, sourcetype 
# retrieve records with, e.g., 'object[["AH5086"]]' 

            title                      
  AH5086  | CpG Islands                
  AH5096  | Evo Cpg                    
  AH5204  | CpG Islands                
  AH5227  | Evo Cpg                    
  AH5344  | CpG Islands                
  ...       ...                        
  AH7168  | CpG Islands                
  AH7203  | CpG Islands                
  AH27911 | PU.1_GSM940934_CpG_BMDM.bw 
  AH28226 | PU.1_GSM940934_CpG_BMDM.bed
  AH28541 | PU.1_GSM940934_CpG_BMDM.csv

In [6]:
# Question1
# 1. 下载 hg38 的 CpG Islands
ah <- AnnotationHub()
cpg <- ah[["AH5086"]]   # hg38 CpG Islands (GRanges)

# 2. 定义 autosomes（1–22号染色体）
autosomes <- paste0("chr", 1:22)

# 3. 过滤出 autosomes 上的 CpG Islands
cpg_autosomes <- cpg[seqnames(cpg) %in% autosomes]

# 4. 统计数量
length(cpg_autosomes)

loading from cache



[1] 26641

# Question2

In [7]:
# Question2
ah <- AnnotationHub()
cpg <- ah[["AH5086"]]   # hg38 CpG Islands

# 过滤出 chr4
cpg_chr4 <- cpg[seqnames(cpg) == "chr4"]

# 统计数量
length(cpg_chr4)

loading from cache



[1] 1031

# Question3

In [12]:
# Question3
res_narrow <- query(ah, c("EpigenomeRoadMap", "H3K4me3", "H1", "narrowPeak"))
res_narrow

AnnotationHub with 26 records
# snapshotDate(): 2026-04-23
# $dataprovider: BroadInstitute
# $species: Homo sapiens
# $rdataclass: GRanges
# additional mcols(): taxonomyid, genome, description,
#   coordinate_1_based, maintainer, rdatadateadded, preparerclass, tags,
#   rdatapath, sourceurl, sourcetype 
# retrieve records with, e.g., 'object[["AH29884"]]' 

            title                                                              
  AH29884 | E003-H3K4me3.narrowPeak.gz                                         
  AH29911 | E004-H3K4me3.narrowPeak.gz                                         
  AH29933 | E005-H3K4me3.narrowPeak.gz                                         
  AH29958 | E006-H3K4me3.narrowPeak.gz                                         
  AH29981 | E007-H3K4me3.narrowPeak.gz                                         
  ...       ...                                                                
  AH44402 | UCSD.H1_Derived_Neuronal_Progenitor_Cultured_Cells.H3K4me3.AK243...


In [13]:
h3k4 <- ah[["AH29884"]]
autosomes <- paste0("chr", 1:22)
h3k4_auto <- h3k4[seqnames(h3k4) %in% autosomes]

loading from cache



In [14]:
total_bases <- sum(width(h3k4_auto))
total_bases

[1] 41135164

# Question4

In [15]:
library(AnnotationHub)
library(GenomicRanges)

ah <- AnnotationHub()

# 一步找到 Roadmap + H1 + H3K27me3 + narrowPeak
res <- query(ah, c("EpigenomeRoadMap", "H3K27me3", "H1", "narrowPeak"))
res

AnnotationHub with 25 records
# snapshotDate(): 2026-04-23
# $dataprovider: BroadInstitute
# $species: Homo sapiens
# $rdataclass: GRanges
# additional mcols(): taxonomyid, genome, description,
#   coordinate_1_based, maintainer, rdatadateadded, preparerclass, tags,
#   rdatapath, sourceurl, sourcetype 
# retrieve records with, e.g., 'object[["AH29892"]]' 

            title                                                              
  AH29892 | E003-H3K27me3.narrowPeak.gz                                        
  AH29917 | E004-H3K27me3.narrowPeak.gz                                        
  AH29940 | E005-H3K27me3.narrowPeak.gz                                        
  AH29965 | E006-H3K27me3.narrowPeak.gz                                        
  AH29988 | E007-H3K27me3.narrowPeak.gz                                        
  ...       ...                                                                
  AH44379 | UCSD.H1_Derived_Mesenchymal_Stem_Cells.H3K27me3.SK439.narrowPeak.gz


In [17]:
h3k27 <- ah[["AH29892"]]
autosomes <- paste0("chr", 1:22)
h3k27_auto <- h3k27[seqnames(h3k27) %in% autosomes]
mean_signal <- mean(mcols(h3k27_auto)$signalValue)
mean_signal

loading from cache



[1] 4.770728

# Question5

In [18]:
# 找到所有 H3K4me3 与 H3K27me3 的重叠对
hits <- findOverlaps(h3k4_auto, h3k27_auto)

# 对每一对重叠区间，计算交集（bivalent 区域）
bivalent <- pintersect(h3k4_auto[queryHits(hits)],
                       h3k27_auto[subjectHits(hits)])

In [19]:
total_bivalent_bases <- sum(width(bivalent))
total_bivalent_bases

[1] 10289096

# Quesion6

In [20]:
## bivalent = intersection of peaks
hits <- findOverlaps(h3k4_auto, h3k27_auto)
bivalent <- pintersect(h3k4_auto[queryHits(hits)],
                       h3k27_auto[subjectHits(hits)])

In [21]:
cpg <- ah[["AH5086"]]   # hg38 CpG Islands
cpg_auto <- cpg[seqnames(cpg) %in% autosomes]

loading from cache



In [22]:
ov <- findOverlaps(bivalent, cpg_auto)

# 哪些 bivalent 区间至少 overlap 一个 CpG Island？
bivalent_with_cpg <- unique(queryHits(ov))

# 计算比例
fraction <- length(bivalent_with_cpg) / length(bivalent)
fraction

[1] 0.5383644

# Question7

In [24]:
## 找到 CpG 与 bivalent 的重叠
ov_cpg_biv <- findOverlaps(cpg_auto, bivalent)

## 对每一对重叠区间求交集（避免用 union 导致重复计数）
cpg_biv_intersect <- pintersect(cpg_auto[queryHits(ov_cpg_biv)],
                                bivalent[subjectHits(ov_cpg_biv)])

## 有可能同一段 CpG 被多个 bivalent 片段重叠，交集会碎片化
## 用 reduce() 合并相邻/重叠片段，避免重复计数碱基
cpg_biv_reduced <- reduce(cpg_biv_intersect)

## 分子：CpG ∩ bivalent 的碱基总数
bases_cpg_biv <- sum(width(cpg_biv_reduced))

## 分母：所有 CpG Islands（autosomes）的碱基总数
bases_cpg_total <- sum(width(cpg_auto))

## 最终比例
fraction_bases_cpg_bivalent <- bases_cpg_biv / bases_cpg_total
fraction_bases_cpg_bivalent

[1] 0.241688

# Question8

In [25]:
## 6. 扩展 CpG Islands ±10kb
cpg_10kb <- resize(cpg_auto, width = width(cpg_auto) + 20000, fix = "center")

## 7. 求交集
ov <- findOverlaps(cpg_10kb, bivalent)
biv_near_cpg <- pintersect(
    cpg_10kb[queryHits(ov)],
    bivalent[subjectHits(ov)]
)

## 8. 合并区间避免重复计数
biv_near_cpg_reduced <- reduce(biv_near_cpg)

## 9. 计算总碱基数
sum(width(biv_near_cpg_reduced))

[1] 9782086

# Question9

In [39]:
cpg_res <- query(ah, "CpG")
cpg_res

AnnotationHub with 62 records
# snapshotDate(): 2026-04-23
# $dataprovider: UCSC, Haemcode
# $species: Homo sapiens, Bos taurus, Pan troglodytes, Mus musculus, Felis c...
# $rdataclass: GRanges, BigWigFile
# additional mcols(): taxonomyid, genome, description,
#   coordinate_1_based, maintainer, rdatadateadded, preparerclass, tags,
#   rdatapath, sourceurl, sourcetype 
# retrieve records with, e.g., 'object[["AH5086"]]' 

            title                      
  AH5086  | CpG Islands                
  AH5096  | Evo Cpg                    
  AH5204  | CpG Islands                
  AH5227  | Evo Cpg                    
  AH5344  | CpG Islands                
  ...       ...                        
  AH7168  | CpG Islands                
  AH7203  | CpG Islands                
  AH27911 | PU.1_GSM940934_CpG_BMDM.bw 
  AH28226 | PU.1_GSM940934_CpG_BMDM.bed
  AH28541 | PU.1_GSM940934_CpG_BMDM.csv

In [40]:
# 1. 获取 hg38 CpG Islands
cpg <- ah[["AH5086"]]

# 2. 计算 CpG Islands 的总碱基数（注意 reduce 去重）
cpg_total_bases <- sum(as.numeric(width(reduce(cpg))))
cpg_total_bases

loading from cache



[1] 21842742

In [41]:
standard_chrs <- paste0("chr", c(1:22, "X", "Y"))

# 只取标准染色体的长度，并转成 numeric，避免整数溢出
genome_total_bases <- sum(
  as.numeric(seqlengths(cpg)[standard_chrs]),
  na.rm = TRUE
)
genome_total_bases

[1] 3095677412

In [42]:
# 4. 计算比例
fraction <- cpg_total_bases / genome_total_bases
fraction

[1] 0.007055884

# Question 10

In [47]:
bivalent_auto <- reduce(bivalent)
bivalent_auto

GRanges object with 22182 ranges and 0 metadata columns:
          seqnames            ranges strand
             <Rle>         <IRanges>  <Rle>
      [1]     chr1     763012-763140      *
      [2]     chr1     839484-839665      *
      [3]     chr1     840268-841052      *
      [4]     chr1     858803-861434      *
      [5]     chr1     870557-870697      *
      ...      ...               ...    ...
  [22178]    chr22 51135342-51135727      *
  [22179]    chr22 51136248-51136589      *
  [22180]    chr22 51158284-51159540      *
  [22181]    chr22 51169377-51169735      *
  [22182]    chr22 51195351-51195903      *
  -------
  seqinfo: 24 sequences from hg19 genome; no seqlengths

In [44]:
cpg_auto

GRanges object with 26641 ranges and 1 metadata column:
          seqnames            ranges strand |        name
             <Rle>         <IRanges>  <Rle> | <character>
      [1]     chr1       28736-29810      * |    CpG:_116
      [2]     chr1     135125-135563      * |     CpG:_30
      [3]     chr1     327791-328229      * |     CpG:_29
      [4]     chr1     437152-438164      * |     CpG:_84
      [5]     chr1     449274-450544      * |     CpG:_99
      ...      ...               ...    ... .         ...
  [26637]    chr22 51135671-51136118      * |     CpG:_44
  [26638]    chr22 51142803-51143308      * |     CpG:_38
  [26639]    chr22 51158387-51160060      * |    CpG:_167
  [26640]    chr22 51169028-51170019      * |     CpG:_81
  [26641]    chr22 51221773-51222317      * |     CpG:_63
  -------
  seqinfo: 93 sequences (1 circular) from hg19 genome

In [49]:
### 4. 计算 A = CpG ∩ bivalent 的碱基数
ov_A <- pintersect(cpg_auto[queryHits(findOverlaps(cpg_auto, bivalent_auto))],
                   bivalent_auto[subjectHits(findOverlaps(cpg_auto, bivalent_auto))])
A <- sum(width(reduce(ov_A)))

### 5. 计算 CpG 总碱基数
CpG_total <- sum(width(reduce(cpg_auto)))

### 6. 计算 bivalent 总碱基数
biv_total <- sum(width(bivalent_auto))

### 7. B = CpG but not bivalent
B <- CpG_total - A

### 8. C = bivalent but not CpG
C <- biv_total - A

### 9. D = genome - (A + B + C)
genome_total <- sum(as.numeric(seqlengths(cpg)[autosomes]))
D <- genome_total - (A + B + C)

### 10. 构建 2×2 表
cont <- matrix(c(A, B, C, D), nrow = 2,
               dimnames = list(
                 Bivalent = c("Yes", "No"),
                 CpG = c("Yes", "No")
               ))
cont

,Yes,No
Yes,4907239,5381857
No,15396787,2855347403


In [52]:
A <- 4907239
B <- 15396787
C <- 5381857
D <- 2855347403

OR <- (A * D) / (B * C)
OR

[1] 169.0962